# Word2Vec Tutorial with Gensim

In [ ]:
!pip install gensim

In [ ]:
# imports

import json
from collections import Counter
from gensim.models import Word2Vec
import matplotlib.pyplot as plt
import utils

%matplotlib inline

In [ ]:
# Load and display data

with open("countries.json", "r") as fout:
    countries = json.load(fout)

In [ ]:
countries["India"][:20]

In [ ]:
print(" ".join(countries["India"])[:512] + " ...")

In [ ]:
for i, (country, text) in enumerate(countries.items()):
    print(country)
    print(" ".join(text)[:512] + " ...")
    print("-" * 100)
    if i >= 5:
        break

## Basic Word2Vec Usage

In [ ]:
# Create and train a simple model

model = Word2Vec(sentences=countries.values())

In [ ]:
# Check word similarities learnt by the model

model.wv.most_similar("India", topn=5)

In [ ]:
# Enable computation of loss

model = Word2Vec(sentences=countries.values(), compute_loss=True)
model.get_latest_training_loss()

### Word2Vec options

In [ ]:
?Word2Vec

## Heuristics for Word2vec algorithms

### Determining size of the vocabulary

In [ ]:
# How many unique words in the vocabulary?

counter = Counter()
for words in countries.values():
    for word in words:
        counter.update([word])

print(len(counter))

In [ ]:
# Default vocabulary size of the original model

len(model.wv)

In [ ]:
# Retrain - increased vocabulary size, more epochs, larger word vectors

metric = utils.MetricCallback(every=1)
model = Word2Vec(
    sentences=countries.values(),
    vector_size=128,
    epochs=10,
    max_vocab_size=65536,
    compute_loss=True,
    callbacks=[metric],
)
plt.plot(metric.myloss)

In [ ]:
# Check similarities again

model.wv.most_similar("India")

In [ ]:
# Retrain - more epochs

metric = utils.MetricCallback(every=10)
model = Word2Vec(
    sentences=countries.values(),
    vector_size=128,
    epochs=100,
    max_vocab_size=65536,
    compute_loss=True,
    callbacks=[metric],
    min_alpha=0.001,
    workers=9,
)
plt.plot(metric.myloss)

In [ ]:
model.wv.most_similar("India")

In [ ]:
# Examine the vector space

X = ["India", "Pakistan", "Bangladesh", "France", "England", "Spain"]
Y = ["Delhi", "Islamabad", "Dhaka", "Paris", "London", "Madrid"]
utils.plot_arrows(X, Y, model.wv)

In [ ]:
# Visualize vectors for all countries

utils.plot_vectors(countries, model)

## Word Analogies

In [ ]:
# India: Ganges -> Brazil: __ ?

model.wv.most_similar(positive=["Ganges", "Brazil"], negative=["India"])

In [ ]:
# America: Washington -> France: __ ?

model.wv.most_similar(positive=["Washington", "France"], negative=["America"])

In [ ]:
# India: Hindi -> Germany: __ ?

model.wv.most_similar(positive=["Hindi", "Germany"], negative=["India"])

In [ ]:
# Save the model

model.save("wiki-countries.w2v")

In [ ]:
from gensim.models import KeyedVectors

model = KeyedVectors.load("wiki-countries.w2v")

In [ ]:
model